# 선형회귀_ 모델링용 전처리

- 전처리 Process
    - 라벨링 -> 더미변수 -> 로그변환 -> 이상치 대체 -> 다중공선성 -> 정규화
    - **순서 꼭 지켜야한다.**

## #01. 준비작업

### 1. 이전 작업 내역 불러오기

In [1]:
%%capture cap
%run "C:/코딩공부/4. 통계/0630/LAB08 [PBT] EDA 시작하기.ipynb"

### 2. 불러온 내용 확인

In [2]:
print("목표변수 :", target)
print("연속형(종속변수 포함) :", continuous_cols)
print("독립변수(명목형): ", nominal_cols)
print("독립변수(연속형): ", feature_continuous)

display(df.head())              # 품질점검 완료 후 타입변환 데이터프레임
display(desc)                   # 기술통계량 데이터프레임
display(cat_desc)               # 범주형 데이터 기술통계량 데이터프레임

목표변수 : MEDV
연속형(종속변수 포함) : ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
독립변수(명목형):  ['CHAS']
독립변수(연속형):  ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,18.000,2.310,0,0.538,6.575,65.200,4.090,1,296,15.300,396.900,4.980,24.000
1,0.027,0.000,7.070,0,0.469,6.421,78.900,4.967,2,242,17.800,396.900,9.140,21.600
2,0.027,0.000,7.070,0,0.469,7.185,61.100,4.967,2,242,17.800,392.830,4.030,34.700
3,0.032,0.000,2.180,0,0.458,6.998,45.800,6.062,3,222,18.700,394.630,2.940,33.400
4,0.069,0.000,2.180,0,0.458,7.147,54.200,6.062,3,222,18.700,396.900,5.330,36.200


,count,mean,std,min,25%,50%,75%,max,rel_diff,rdiff_flag,iqr,upper_bound,lower_bound,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio,skew,skew_interpret,kurt,kurt_interpret,log_need
CRIM,506.000,3.614,8.602,0.006,0.082,0.257,3.677,88.976,13.087,large_diff,3.595,9.070,-5.311,66,0.130,0,0.000,66,0.130,5.223,right tail,37.131,leptokurtic,log1p
ZN,506.000,11.364,23.322,0.000,0.000,0.000,12.500,100.000,inf,large_diff,12.500,31.250,-18.750,68,0.134,0,0.000,68,0.134,2.226,right tail,4.032,leptokurtic,log1p
INDUS,506.000,11.137,6.860,0.460,5.190,9.690,18.100,27.740,0.149,diff,12.910,37.465,-14.175,0,0.000,0,0.000,0,0.000,0.295,symmetric,-1.234,platykurtic,none
NOX,506.000,0.555,0.116,0.385,0.449,0.538,0.624,0.871,0.031,similar,0.175,0.886,0.187,0,0.000,0,0.000,0,0.000,0.729,right tail,-0.065,platykurtic,none
RM,506.000,6.285,0.703,3.561,5.885,6.208,6.623,8.780,0.012,similar,0.738,7.731,4.778,22,0.043,8,0.016,30,0.059,0.404,symmetric,1.892,leptokurtic,none
AGE,506.000,68.575,28.149,2.900,45.025,77.500,94.075,100.000,0.115,diff,49.050,167.650,-28.550,0,0.000,0,0.000,0,0.000,-0.599,left tail,-0.968,platykurtic,none
DIS,506.000,3.795,2.106,1.130,2.100,3.207,5.188,12.127,0.183,diff,3.088,9.821,-2.532,5,0.010,0,0.000,5,0.010,1.012,right tail,0.488,leptokurtic,log1p
RAD,506.000,9.549,8.707,1.000,4.000,5.000,24.000,24.000,0.910,large_diff,20.000,54.000,-26.000,0,0.000,0,0.000,0,0.000,1.005,right tail,-0.867,platykurtic,log1p
TAX,506.000,408.237,168.537,187.000,279.000,330.000,666.000,711.000,0.237,diff,387.000,1246.500,-301.500,0,0.000,0,0.000,0,0.000,0.670,right tail,-1.142,platykurtic,none
PTRATIO,506.000,18.456,2.165,12.600,17.400,19.050,20.200,22.000,0.031,similar,2.800,24.400,13.200,0,0.000,15,0.030,15,0.030,-0.802,left tail,-0.285,platykurtic,none


,CHAS
count,506
unique,2
top,0
freq,471


## #my_prep.py 모듈은 1교시에 작성함

## #02. 데이터 전처리

### 1. 명목형 라벨링
- 명목형 컬럼(CHAS)은 category 타입이라 그대로는 모델에 넣을 수 없어 정수로 바꿈

In [3]:
df1 = my_prep.labeling(df, columns=nominal_cols)
df1.head()

CHAS (2종): 0=0, 1=1


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,18.000,2.310,0,0.538,6.575,65.200,4.090,1,296,15.300,396.900,4.980,24.000
1,0.027,0.000,7.070,0,0.469,6.421,78.900,4.967,2,242,17.800,396.900,9.140,21.600
2,0.027,0.000,7.070,0,0.469,7.185,61.100,4.967,2,242,17.800,392.830,4.030,34.700
3,0.032,0.000,2.180,0,0.458,6.998,45.800,6.062,3,222,18.700,394.630,2.940,33.400
4,0.069,0.000,2.180,0,0.458,7.147,54.200,6.062,3,222,18.700,396.900,5.330,36.200


#### 명목형 변수의 One-Hot-Encoding(더미변수)
- 명목형 데이터의 수치형 데이터 변환
    : 라벨링만 적용하면 모델이 숫자의 크기에 잘못된 순서.서열을 학습할 수 있어 **독립적인 더미변수로 변환**해주는 처리
    - 통계적 용어: 더미변수 / 머신러닝 용어: One-Hot-Encoding(혹은 Encoding)

| 성별 |
---------
| 남자 |
---------
| 여자 | 

-> 라벨링 (숫자로 값을 구분)

| 성별 |
---------
| 0 |
---------
| 1 | 

-> 인코딩 (라벨링된 값은 컬럼명의 접미사가 됨)

| 성별_0 | 성별_1 |
-------------------
|    1   |    0   |
-------------------
|    0   |    1   |
----------------------------------------------------
- 라벨링을 생략한 One-Hot-Encoding
    - 인코딩을 위해 라벨링이 반드시 필요한 것은 아님
    - 라벨링을 거치지 않을 경우 변수 이름이 지나치게 길고 복잡해짐 (성별_남자/ 성별_여자)
    * pandas 패키지를 사용하는 방법과 scikit-learn 패키지 사용하는 방법이 있음
        * 통계 : pandas / 머신러닝 : scikit-learn 사용

#### 더미 변수 처리의 drop_first=True
- 이진 변수 -> 값이 0과 1로만 구성되어 있는 경우
    - 성별_0이 1이라면 (성별이 남자라면), 성별_1은 반드시 0(여자가 아님)이 된다.
        - 라벨링을 수행한 이진 변수의 더미 변수 처리(drop_first=True)는 의미 없는 처리가 됨
| 성별 |
---------
| 남자 |
---------
| 여자 | 

-> 라벨링

| 성별 |
---------
| 0 |
---------
| 1 | 

-> 인코딩 (drop_first=True)

| 성별_0 | 성별_1 |
-------------------
|    1   |    0   |
-------------------
|    0   |    1   | => 왼쪽 컬럼값은 의미가 없어짐

- 값의 종류가 3개 이상인 경우
    - 더미 변수 처리는 값의 종류별로 컬럼(변수)을 만들고 이 중 **하나가 1이라면 나머지 항목은 필연적으로 0이 되므로** 첫 번째 항목을 제거하더라도 충분히 추론 가능
        - 합격 여부가 보류인 경우 합격여부_0이 1이므로 나머지는 반드시 0이 된다.


### 2. 더미변수 인코딩
- 라벨링된 명목형 컬럼을 더미 변수로 펼친다
    - 이 과정까지는 **모델에 넣기 위한 필수 변환**, 이 시점의 데이터프레임을 **기준선으로 저장**

In [4]:
df2 = my_prep.dummies(df1, columns=nominal_cols, drop_first=True)
df2.to_excel("체크포인트0_기준선.xlsx", index=False)
df2.head()

CHAS (2종) -> 생략: 이진 변수이므로 원래 컬럼을 유지

컬럼 수: 14개 -> 14개


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,18.000,2.310,0,0.538,6.575,65.200,4.090,1,296,15.300,396.900,4.980,24.000
1,0.027,0.000,7.070,0,0.469,6.421,78.900,4.967,2,242,17.800,396.900,9.140,21.600
2,0.027,0.000,7.070,0,0.469,7.185,61.100,4.967,2,242,17.800,392.830,4.030,34.700
3,0.032,0.000,2.180,0,0.458,6.998,45.800,6.062,3,222,18.700,394.630,2.940,33.400
4,0.069,0.000,2.180,0,0.458,7.147,54.200,6.062,3,222,18.700,396.900,5.330,36.200


### 3. 로그 변환 대상 컬럼 선정

In [5]:
# 우측꼬리 로그 변환 대상 컬럼
log_cols = desc.loc[desc['log_need'] == 'log1p'].index.tolist()
log_cols

['CRIM', 'ZN', 'DIS', 'RAD', 'LSTAT', 'MEDV']

In [7]:
# 좌측꼬리 로그 변환 대상 컬럼
reflect_cols = desc.loc[desc['log_need'] == 'reverse_log1p'].index.tolist()
reflect_cols

['B']

### 4. 로그 변환

In [9]:
df3 = my_prep.log_transform(df2, log_columns=log_cols,
                            reflect_columns=reflect_cols)
df3.to_excel("체크포인트1_로그변환.xlsx", index=False)
df3.head()

컬럼        꼬리방향      변환식                   역변환식                                  왜도
----------------------------------------------------------------------------------------
CRIM      우측 꼬리     log(1+x)              exp(y)-1                  5.21 -> 1.27
ZN        우측 꼬리     log(1+x)              exp(y)-1                  2.22 -> 1.19
DIS       우측 꼬리     log(1+x)              exp(y)-1                  1.01 -> 0.33
RAD       우측 꼬리     log(1+x)              exp(y)-1                  1.00 -> 0.53
LSTAT     우측 꼬리     log(1+x)              exp(y)-1                 0.90 -> -0.19
MEDV      우측 꼬리     log(1+x)              exp(y)-1                 1.10 -> -0.24
B         좌측 꼬리     log(1+396.9-x)        396.9-(exp(y)-1)         -2.88 -> 0.60


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,2.944,2.310,0,0.538,6.575,65.200,1.627,0.693,296,15.300,0.000,1.788,3.219
1,0.027,0.000,7.070,0,0.469,6.421,78.900,1.786,1.099,242,17.800,0.000,2.316,3.118
2,0.027,0.000,7.070,0,0.469,7.185,61.100,1.786,1.099,242,17.800,1.623,1.615,3.575
3,0.032,0.000,2.180,0,0.458,6.998,45.800,1.955,1.386,222,18.700,1.185,1.371,3.538
4,0.067,0.000,2.180,0,0.458,7.147,54.200,1.955,1.386,222,18.700,0.000,1.845,3.616


In [ ]:
# 좌측꼬리 로그 변환 대상 컬럼
reflect_cols = desc.loc[desc['log_need'] == 'reverse_log1p'].index.tolist()
reflect_cols

['B']

#### 모델링 직전 이상치 처리 결정
- 이상치를 처리하지 않으면 원본 데이터가 꼬리가 있을 확률이 높다.
    - 선형회귀를 그렸을 때 이상치가 보이게 되면 예측이 틀어지는 경우가 있다.
- 일정 수준의 값들은 잘라내는 것이 좋음 clip()
- 이상치를 평균이나 중앙값, 경계값으로 바꾸는것을 추천한다. 특히 **경계값**으로
- 잔차가 몰려버리는 구간들은 '한계점'으로 명시한다

### 5. 이상치 대체
- IQR 울타리(`Q1 - 1.5 * IQR Q3 + 1.5 * IQR`)을 벗어난 값 = 경계값
    - 행을 삭제하지 않으므로 데이터 수 (506개)가 그대로 유지
- 경계값(`bound`)을 쓰는 이유는 값의 순서가 보존되기 때문
    - 중앙값으로 대체하면 범죄율이 가장 높은 동네가 가장 안전한 동네로 뒤바뀌어 버리는 경우도 있다


In [10]:
df4 = my_prep.replace_outlier(df3, columns=feature_continuous)
df4.to_excel("체크포인트2_이상치대체.xlsx", index=False)
df4.head()

이상치 대체 방식: 'bound' (기준: IQR x 1.5)
컬럼                               정상 범위       이상치                   대체값
--------------------------------------------------------------------
CRIM                    -2.12 ~ 3.74    6개(1.2%)       -2.12 또는 3.74
ZN                      -3.90 ~ 6.51    0개(0.0%)       -3.90 또는 6.51
INDUS                 -14.18 ~ 37.47    0개(0.0%)     -14.18 또는 37.47
NOX                      0.19 ~ 0.89    0개(0.0%)        0.19 또는 0.89
RM                       4.78 ~ 7.73   30개(5.9%)        4.78 또는 7.73
AGE                  -28.55 ~ 167.65    0개(0.0%)    -28.55 또는 167.65
DIS                      0.09 ~ 2.86    0개(0.0%)        0.09 또는 2.86
RAD                     -0.80 ~ 5.63    0개(0.0%)       -0.80 또는 5.63
TAX                -301.50 ~ 1246.50    0개(0.0%)  -301.50 또는 1246.50
PTRATIO                13.20 ~ 24.40   15개(3.0%)      13.20 또는 24.40
B                       -3.38 ~ 7.01    0개(0.0%)       -3.38 또는 7.01
LSTAT                    0.85 ~ 4.11    0개(0.0%)        0.85 또는 4.

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,2.944,2.310,0,0.538,6.575,65.200,1.627,0.693,296,15.300,0.000,1.788,3.219
1,0.027,0.000,7.070,0,0.469,6.421,78.900,1.786,1.099,242,17.800,0.000,2.316,3.118
2,0.027,0.000,7.070,0,0.469,7.185,61.100,1.786,1.099,242,17.800,1.623,1.615,3.575
3,0.032,0.000,2.180,0,0.458,6.998,45.800,1.955,1.386,222,18.700,1.185,1.371,3.538
4,0.067,0.000,2.180,0,0.458,7.147,54.200,1.955,1.386,222,18.700,0.000,1.845,3.616


### 6. 다중 공선성 제거
- VIF가 임계값을 넘는 변수를 하나씩 제거
- 종속변수(MEDV)는 대상에서 빼야 하므로 연속형 독립변수만 지정

In [11]:
df5 = my_prep.reduce_vif(df4, columns=feature_continuous, threshold=10.0)
df5.to_excel("체크포인트3_다중공선성제거.xlsx", index=False)
df5.head()


완료! 남은 변수: ['AGE', 'B', 'CRIM', 'DIS', 'INDUS', 'LSTAT', 'NOX', 'PTRATIO', 'RAD', 'RM', 'TAX', 'ZN']
최대 VIF = 7.08


,AGE,B,CRIM,DIS,INDUS,LSTAT,NOX,PTRATIO,RAD,RM,TAX,ZN
0,65.200,0.000,0.006,1.627,2.310,1.788,0.538,15.300,0.693,6.575,296,2.944
1,78.900,0.000,0.027,1.786,7.070,2.316,0.469,17.800,1.099,6.421,242,0.000
2,61.100,1.623,0.027,1.786,7.070,1.615,0.469,17.800,1.099,7.185,242,0.000
3,45.800,1.185,0.032,1.955,2.180,1.371,0.458,18.700,1.386,6.998,222,0.000
4,54.200,0.000,0.067,1.955,2.180,1.845,0.458,18.700,1.386,7.147,222,0.000


### 7. 정규화
- 연속형 독립변수를 대상
- 정규화는 OLS의 R squared를 바꾸지 못함

In [12]:
df6 = my_prep.scaling(df5, columns=feature_continuous, method='standard')
df6.to_excel("체크포인트4_정규화.xlsx", index=False)
df6.head()

StandardScaler 적용 (12개 컬럼)
컬럼                            변환 전                  변환 후
--------------------------------------------------------
CRIM                 0.01 ~ 3.74          -0.80 ~ 2.91
ZN                   0.00 ~ 4.62          -0.59 ~ 2.26
INDUS               0.46 ~ 27.74          -1.56 ~ 2.42
NOX                  0.39 ~ 0.87          -1.47 ~ 2.73
RM                   4.78 ~ 7.73          -2.38 ~ 2.31
AGE                2.90 ~ 100.00          -2.34 ~ 1.12
DIS                  0.76 ~ 2.57          -1.75 ~ 2.65
RAD                  0.69 ~ 3.22          -1.81 ~ 1.55
TAX              187.00 ~ 711.00          -1.31 ~ 1.80
PTRATIO            13.20 ~ 22.00          -2.46 ~ 1.65
B                    0.00 ~ 5.99          -1.18 ~ 2.23
LSTAT                1.00 ~ 3.66          -2.73 ~ 2.20


,AGE,B,CRIM,DIS,INDUS,LSTAT,NOX,PTRATIO,RAD,RM,TAX,ZN
0,-0.120,-1.184,-0.796,0.357,-1.288,-1.276,-0.144,-1.477,-1.812,0.476,-0.667,1.229
1,0.367,-1.184,-0.776,0.742,-0.593,-0.295,-0.740,-0.310,-1.273,0.231,-0.987,-0.589
2,-0.266,-0.257,-0.776,0.742,-0.593,-1.597,-0.740,-0.310,-1.273,1.445,-0.987,-0.589
3,-0.810,-0.507,-0.771,1.150,-1.307,-2.051,-0.835,0.110,-0.890,1.148,-1.106,-0.589
4,-0.511,-1.184,-0.736,1.150,-1.307,-1.170,-0.835,0.110,-0.890,1.384,-1.106,-0.589
